# Разовая загрузка амортизации терминалов в озеро

Цель:
- взять файл `term_mod_id.xlsx` (`c_nter + model`),
- нормализовать модель,
- назначить амортизацию по правилам (из фото),
- разово загрузить terminal-level таблицу в `sandbox`.

Выход:
- `sandbox_ai.shestopalov_terminal_amortization_oneoff`

In [ ]:
import os
import re
import subprocess
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры разовой загрузки
excel_path = '/home/jovyan/documents/Equaring/Data/term_mod_id.xlsx'
source_file = 'term_mod_id.xlsx'

# Целевая таблица в sandbox_ai, имя начинается с shestopalov_
target_table = 'sandbox_ai.shestopalov_terminal_amortization_oneoff'

# Локальный файл для выгрузки в ORC + загрузки в HDFS
save_table_orc_name = 'shestopalov_terminal_amortization_oneoff.orc'

print('excel_path =', excel_path)
print('target_table =', target_table)
print('save_table_orc_name =', save_table_orc_name)

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

dl = connect(
    to='DATALAKE',
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
dl._init_connection()

print('Impala + Datalake initialized')

## 1) Загрузка Excel и нормализация (`c_nter`, `model`)

In [ ]:
def normalize_c_nter(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    return s.upper()

def normalize_model(v):
    if pd.isna(v):
        return None
    s = str(v).strip().lower().replace('\xa0', ' ')
    s = s.replace('ё', 'е')
    s = re.sub(r"\(.*?\)", ' ', s)
    s = re.sub(r"[^a-zа-я0-9+ ]", ' ', s)
    s = re.sub(r"\s+", ' ', s).strip()
    if s == '' or s in {'nan', 'none', 'null'}:
        return None
    return s

raw = pd.read_excel(excel_path)
raw.columns = [str(c).strip() for c in raw.columns]

# Поддержка ожидаемых имен из файла
col_model_candidates = ['Модель', 'model', 'MODEL']
col_nter_candidates = ['ID терминала', 'c_nter', 'ID_TERMINAL', 'terminal_id']

model_col = next((c for c in col_model_candidates if c in raw.columns), None)
nter_col = next((c for c in col_nter_candidates if c in raw.columns), None)

if model_col is None or nter_col is None:
    raise ValueError(f'Cannot detect required columns. columns={raw.columns.tolist()}')

stg_term_model = raw[[nter_col, model_col]].copy()
stg_term_model.columns = ['c_nter_raw', 'model_raw']
stg_term_model['c_nter'] = stg_term_model['c_nter_raw'].apply(normalize_c_nter)
stg_term_model['model_norm'] = stg_term_model['model_raw'].apply(normalize_model)

stg_term_model = stg_term_model.dropna(subset=['c_nter']).copy()

print('rows_raw =', len(raw))
print('rows_stg =', len(stg_term_model))
print('distinct_c_nter =', stg_term_model['c_nter'].nunique())
display(stg_term_model.head(50))

## 2) Правила амортизации по модели (из фото) + расчет флагов

In [ ]:
# Правила из фото (в месяц)
# normalized_model_pattern -> amortization_monthly
amortization_rules = [
    ('pax s100', 289.61),
    ('d230 e4gbtwifi', 104.11),
    ('d230 4gbtwifi', 354.95),
    ('d230 4gbtwifi+b', 354.95),
    ('pax s200cl', 104.11),
    ('lifepay s10f', 532.46),
    ('inpas a8900', 348.48),
    ('fujian morefun mf360', 270.21),
    ('sommers p180', 804.79),
]

def get_amortization_by_model(model_norm):
    if model_norm is None or pd.isna(model_norm):
        return 0.0, 1
    m = str(model_norm)
    for pattern, amt in amortization_rules:
        if pattern in m:
            return float(amt), 0
    return 0.0, 1

tmp = stg_term_model['model_norm'].apply(get_amortization_by_model)
stg_term_model['amortization_monthly'] = tmp.apply(lambda x: x[0])
stg_term_model['rule_not_found'] = tmp.apply(lambda x: x[1])
stg_term_model['is_amortized'] = stg_term_model['amortization_monthly'].apply(lambda x: 1 if x > 0 else 0)
stg_term_model['source_file'] = source_file
stg_term_model['load_dt'] = pd.Timestamp.now().normalize()

display(stg_term_model.head(100))

## 3) Разовая загрузка в sandbox (staging -> target)

In [ ]:
# Загрузка по шаблону: df -> ORC -> HDFS -> external table

def load_to_datalake_from_file(local_file_path: str, table: str, dest_catalog: str = 'external'):
    schema = table.split('.')[0]
    table_name = table.split('.')[-1]
    hdfs_path = f'/warehouse/tablespace/{dest_catalog}/hive/{schema}.db/{table_name}'

    subprocess.run(['hdfs', 'dfs', '-copyFromLocal', '-f', local_file_path, f'{hdfs_path}/{local_file_path}'], check=True)
    out = subprocess.run(['hdfs', 'dfs', '-ls', '-h', hdfs_path], capture_output=True, text=True, check=True)
    print(f'Files in hdfs by path {hdfs_path}:\n{out.stdout}')

# Готовим df для выгрузки
to_load = stg_term_model[
    ['c_nter', 'model_raw', 'model_norm', 'amortization_monthly', 'is_amortized', 'rule_not_found', 'load_dt', 'source_file']
].copy()

to_load['load_dt'] = pd.to_datetime(to_load['load_dt']).dt.strftime('%Y-%m-%d')
to_load = to_load.fillna({'model_raw': '', 'model_norm': '', 'amortization_monthly': 0.0, 'is_amortized': 0, 'rule_not_found': 1, 'source_file': source_file})

to_load.to_orc(save_table_orc_name, index=False)

with dl:
    dl.execute(f'''drop table if exists {target_table}''')
    dl.execute(f'''
        create external table if not exists {target_table} (
            c_nter string,
            model_raw string,
            model_norm string,
            amortization_monthly double,
            is_amortized int,
            rule_not_found int,
            load_dt string,
            source_file string
        )
        stored as orc
        tblproperties ('transactional'='false')
    ''')

load_to_datalake_from_file(save_table_orc_name, target_table)

with imp:
    imp.execute(f'invalidate metadata {target_table}')
    imp.execute(f'refresh {target_table}')

print('One-off load completed:', target_table)

## 4) DQ-сводка после загрузки

In [ ]:
sql_dq = f"""
with base as (
    select *
    from {target_table}
),
dup as (
    select c_nter, count(*) as cnt
    from base
    group by c_nter
    having count(*) > 1
)
select
    (select count(*) from base) as rows_cnt,
    (select count(distinct c_nter) from base) as distinct_c_nter_cnt,
    (select count(*) from dup) as duplicated_c_nter_cnt,
    (select sum(case when rule_not_found = 1 then 1 else 0 end) from base) as rule_not_found_cnt,
    (select round(100.0 * sum(case when rule_not_found = 1 then 1 else 0 end) / count(*), 2) from base) as rule_not_found_pct,
    (select sum(case when is_amortized = 1 then 1 else 0 end) from base) as is_amortized_cnt
"""

with imp:
    dq_summary = imp.fetch(sql_dq)
    unknown_models = imp.fetch(f"""
        select model_norm, count(*) as cnt
        from {target_table}
        where rule_not_found = 1
        group by model_norm
        order by cnt desc, model_norm
    """)
    dup_c_nter = imp.fetch(f"""
        select c_nter, count(*) as cnt
        from {target_table}
        group by c_nter
        having count(*) > 1
        order by cnt desc, c_nter
    """)
    sample_loaded = imp.fetch(f"""
        select *
        from {target_table}
        order by c_nter
        limit 200
    """)

display(dq_summary)
display(unknown_models.head(200))
display(dup_c_nter.head(200))
display(sample_loaded.head(200))